In [82]:
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import PandasTools, AllChem
import pandas as pd
import subprocess, re, shutil

In [99]:
WORK = Path(r"C:\Users\czywg\Desktop\Programmer_czy\xtb_work")
XTB = Path(r"C:\Users\czywg\miniconda3\envs\xtb\Library\bin\xtb.exe")

def run_xtb(folder, xyz, *flags):
    with open(folder / "xtb.out", "w") as f:
        p = subprocess.run([str(XTB), xyz, *flags, "P", "4", "--json"],
                           cwd = folder, stdout = f, stderr = subprocess.STDOUT)
        return p.returncode

def read_energy(folder):
    text = (folder / "xtbout.json").read_text(encoding = "utf-8")
    return float(re.search(r'"total energy"\s*:\s*(-?[\d.eE+-]+)', text).group(1))

def select_best(md_id, mol):
    md = WORK / md_id
    md.mkdir(exist_ok = True)

    mol = Chem.AddHs(mol)
    para = AllChem.ETKDGv3()
    para.randomSeed = 7917
    fail1 = 0
    while fail1 < 5 and AllChem.EmbedMultipleConfs(mol, numConfs = 10, params = para) == 0:
        para.useRandomCoords = True
        fail1 += 1
    if fail1 == 5:
        raise RuntimeError(f"{md_id}: RDKit 重试 5 次仍无法生成 3D 构象")

    for i in mol.GetConformers():
        cid = i.GetId()
        Chem.MolToXYZFile(mol, str(md / f"conf_{cid:03d}.xyz"), confId = cid)
    
    fail2 = 0
    for i in sorted(md.glob("conf_*.xyz")):
        sub = md / i.stem
        sub.mkdir(exist_ok = True)
        i.replace(sub / "conf.xyz")
        rc = run_xtb(sub, "conf.xyz", "--gfn", "2", "--chrg", "0", "--uhf", "0", "--opt")
        if rc != 0:
            fail2 += 1
        if fail2 >= 3:
            raise RuntimeError(f"{md_id}: 已有 {fail2} 个构象 xtb 优化失败，放弃该分子")
    
    success = [x for x in md.glob("conf_*") if (x / "xtbout.json").exists()]
    best = min(success, key = lambda x : read_energy(x))
    print(best.name, read_energy(best))
    shutil.copy2(md / best.name / "xtbopt.xyz", md / "parent.xyz")

In [100]:
file = Path.cwd().parent / "data" / "test.xlsx"
df = pd.read_excel(file)
df = df.sample(frac = 1, random_state = 114514, ignore_index = True)
PandasTools.AddMoleculeColumnToFrame(df, "SMILES", "ROMol", False)
bad = []
for i in range(len(df)):
    md_id = f"id_{i:03d}"
    if (WORK / md_id / "parent.xyz").exists():
        continue
    try:
        select_best(md_id, df.loc[i, "ROMol"])
    except Exception as err:
        print(f"[{i+1}/{len(df)}] {md_id} 失败: {err}", flush = True)
        bad.append(md_id)
        pd.DataFrame({"id": bad}).to_csv(WORK / "bad_ids.csv", index = False)